In [1]:
import sys, json
sys.path.append("..")
import numpy as np
import pandas as pd

y = pd.read_parquet("../data/processed/y_train.parquet")["_MICHD"].values
oof = pd.read_parquet("../data/processed/oof_probabilities.parquet")["xgb"].values
d = json.load(open("../reports/final_decision.json"))
t_high, t_mod = d["flag_threshold"], d["moderate_threshold"]

band = np.where(oof < t_mod, "Low", np.where(oof < t_high, "Moderate", "High"))
out = {"prevalence": float(y.mean()),
       "thresholds": {"moderate": t_mod, "high": t_high},
       "bands": {}}
for b in ["Low", "Moderate", "High"]:
    m = band == b
    out["bands"][b] = {"share_of_people": float(m.mean()),
                       "share_of_cases": float(y[m].sum() / y.sum()),
                       "observed_rate": float(y[m].mean())}

json.dump(out, open("../reports/risk_bands.json", "w"), indent=2)
print(json.dumps(out, indent=2))

{
  "prevalence": 0.09357103193349743,
  "thresholds": {
    "moderate": 0.05134836211800575,
    "high": 0.0925072431564331
  },
  "bands": {
    "Low": {
      "share_of_people": 0.5301142909238585,
      "share_of_cases": 0.1,
      "observed_rate": 0.01765110534379788
    },
    "Moderate": {
      "share_of_people": 0.1318254777316415,
      "share_of_cases": 0.1,
      "observed_rate": 0.07098099209925184
    },
    "High": {
      "share_of_people": 0.3380602313445,
      "share_of_cases": 0.8,
      "observed_rate": 0.22143043933054393
    }
  }
}
